In [7]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from difflib import get_close_matches
import requests

dataset = pd.read_csv(r"C:\Users\Saksham Awasthi\Downloads\movies.csv")
rating = pd.read_csv(r"C:\Users\Saksham Awasthi\Downloads\ratings.csv")

avg_rating = (
    rating.groupby("movieId")["rating"].mean().reset_index()
)

avg_rating.rename(columns = {"rating": "Average_Rating"},inplace = True)

dataset = dataset.merge(
    avg_rating , on = "movieId",how = "left"
)

dataset['genres'] = dataset['genres'].fillna("")
dataset['genres'] = dataset['genres'].str.replace('|',' ',regex = False)
dataset['title'] = dataset['title'].fillna("")

cv = CountVectorizer(stop_words = 'english')
vectors = cv.fit_transform(dataset['genres']).toarray()

similarity = cosine_similarity(vectors)

def recommend_movies(movie_name):
     
     dataset['title'] = dataset['title'].str.lower()
     movie_name = movie_name.lower()

     if movie_name not in dataset["title"].values:

          close_match = get_close_matches(
               movie_name, 
               dataset['title'].tolist(),
               n = 5,
               cutoff = 0.5)

          if close_match:
            return f"Movie Not Found.\n Did You Mean ?\n".join(close_match)   

          else :
           return "Movie Not Found."  
          
          
     movie_index = dataset[dataset['title'] == movie_name].index[0]
     score = list(enumerate(similarity[movie_index]))
     
     sorted_score = sorted(score,key = lambda x: x[1],reverse = True)
    
     top_movies = sorted_score[1:6]

     
     print("Recommend Movie:\n")

     recommendations = [] 
     for movie in top_movies:
          movie_title = dataset.iloc[movie[0]]['title']
          movie_genre = dataset.iloc[movie[0]]['genres']
          movie_rating = dataset.iloc[movie[0]]["Average_Rating"]

          print(f"Movie : {movie_title.title()}")
          print(f"Genre: {movie_genre}")
          print("-"*40)

          recommendations.append({
             "title": movie_title,
             "genre":movie_genre,
             "rating":round(movie_rating,1)
             })
          
     return recommendations

recommend_movies("Toy Story (1995)")


Recommend Movie:

Movie : Antz (1998)
Genre: Adventure Animation Children Comedy Fantasy
----------------------------------------
Movie : Toy Story 2 (1999)
Genre: Adventure Animation Children Comedy Fantasy
----------------------------------------
Movie : Adventures Of Rocky And Bullwinkle, The (2000)
Genre: Adventure Animation Children Comedy Fantasy
----------------------------------------
Movie : Emperor'S New Groove, The (2000)
Genre: Adventure Animation Children Comedy Fantasy
----------------------------------------
Movie : Monsters, Inc. (2001)
Genre: Adventure Animation Children Comedy Fantasy
----------------------------------------


[{'title': 'antz (1998)',
  'genre': 'Adventure Animation Children Comedy Fantasy',
  'rating': np.float64(3.2)},
 {'title': 'toy story 2 (1999)',
  'genre': 'Adventure Animation Children Comedy Fantasy',
  'rating': np.float64(3.9)},
 {'title': 'adventures of rocky and bullwinkle, the (2000)',
  'genre': 'Adventure Animation Children Comedy Fantasy',
  'rating': np.float64(2.2)},
 {'title': "emperor's new groove, the (2000)",
  'genre': 'Adventure Animation Children Comedy Fantasy',
  'rating': np.float64(3.7)},
 {'title': 'monsters, inc. (2001)',
  'genre': 'Adventure Animation Children Comedy Fantasy',
  'rating': np.float64(3.9)}]